In [1]:
!pip install requests beautifulsoup4

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# -*- coding: utf-8 -*-
"""
네이버 금융 종목 뉴스 크롤러
- 대상 종목의 뉴스 목록 -> 개별 기사 본문 수집 -> JSON 저장
- 1주차 할일 2번: 뉴스 200~300건 수집
"""

import requests
from bs4 import BeautifulSoup
import re
import time
import json

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://finance.naver.com/item/main.naver",
}
REQUEST_DELAY = 1.0  # 서버 부담을 줄이기 위한 요청 간격(초)
ARTICLES_PER_STOCK = 30  # 종목당 목표 수집 건수
MIN_CONTENT_LENGTH = 100  # 이보다 짧으면 단신으로 간주하고 제외

# 확정된 10개 종목 (코스피200 상위 섹터 중복 제외)
STOCKS = {
    "005930": "삼성전자",        # 반도체(메모리)
    "000660": "SK하이닉스",      # 반도체(메모리)
    "005380": "현대차",          # 자동차
    "373220": "LG에너지솔루션",  # 2차전지
    "207940": "삼성바이오로직스", # 바이오/CDMO
    "105560": "KB금융",          # 금융(은행)
    "329180": "HD현대중공업",    # 조선
    "012450": "한화에어로스페이스", # 방산
    "034020": "두산에너빌리티",   # 원전/발전
    "035420": "NAVER",           # 인터넷 플랫폼
}


def clean_text(text: str) -> str:
    """공백/개행 정리"""
    return re.sub(r"\s+", " ", text).strip()


def get_news_list(code: str, pages: int = 5):
    """종목 뉴스 목록 페이지에서 (제목, 링크, 날짜) 수집"""
    links = []
    for page in range(1, pages + 1):
        url = f"https://finance.naver.com/item/news_news.naver?code={code}&page={page}"
        res = requests.get(url, headers=HEADERS)
        res.encoding = "euc-kr"
        soup = BeautifulSoup(res.text, "html.parser")

        # 뉴스 목록은 table.type5 안에 tr로 구성됨 (구조 바뀌면 여기부터 확인)
        rows = soup.select("table.type5 tbody tr")
        for row in rows:
            title_tag = row.select_one("td.title a")
            date_tag = row.select_one("td.date")
            if not title_tag:
                continue

            href = title_tag.get("href", "")
            full_url = "https://finance.naver.com" + href if href.startswith("/") else href

            links.append({
                "title": clean_text(title_tag.get_text()),
                "url": full_url,
                "date": clean_text(date_tag.get_text()) if date_tag else None,
            })

        time.sleep(REQUEST_DELAY)
    return links


def get_article_content(url: str):
    """개별 기사 페이지에서 본문 텍스트 추출"""
    res = requests.get(url, headers=HEADERS)
    res.encoding = "euc-kr"

    # news_read.naver는 본문 없이 JS로 n.news.naver.com으로 리다이렉트만 함
    redirect_match = re.search(r"location\.href\s*=\s*'([^']+)'", res.text)
    if redirect_match:
        real_url = redirect_match.group(1)
        res = requests.get(real_url, headers=HEADERS)
        res.encoding = "utf-8"

    soup = BeautifulSoup(res.text, "html.parser")

    # iframe으로 본문을 로드하는 구버전 케이스도 대비
    iframe = soup.select_one("iframe#news_read_iframe, iframe#mainFrame")
    if iframe and iframe.get("src"):
        iframe_url = iframe["src"]
        if iframe_url.startswith("/"):
            iframe_url = "https://finance.naver.com" + iframe_url
        res = requests.get(iframe_url, headers=HEADERS)
        res.encoding = "utf-8"
        soup = BeautifulSoup(res.text, "html.parser")

    # 본문 후보 셀렉터 (네이버뉴스 통합형 / 금융뉴스 자체 페이지)
    body = (
        soup.select_one("#dic_area")
        or soup.select_one("#articleBodyContents")
        or soup.select_one("#content")
    )
    if not body:
        return None

    for tag in body.select("script, style"):
        tag.decompose()

    return clean_text(body.get_text())


def crawl():
    all_articles = []
    article_id = 0

    for code, name in STOCKS.items():
        print(f"[{name}] 뉴스 목록 수집 중...")
        news_list = get_news_list(code, pages=5)
        collected = 0

        for item in news_list:
            if collected >= ARTICLES_PER_STOCK:
                break
            if not item["url"]:
                continue

            content = get_article_content(item["url"])
            if not content or len(content) < MIN_CONTENT_LENGTH:
                continue

            article_id += 1
            all_articles.append({
                "id": article_id,
                "title": item["title"],
                "content": content,
                "date": item["date"],
                "stock_name": name,
                "stock_code": code,
                "url": item["url"],
            })
            collected += 1
            time.sleep(REQUEST_DELAY)

        print(f"  -> {collected}건 수집")

    return all_articles


if __name__ == "__main__":
    articles = crawl()
    with open("news_data.json", "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)
    print(f"\n총 {len(articles)}건 저장 완료 -> news_data.json")

[삼성전자] 뉴스 목록 수집 중...
  -> 30건 수집
[SK하이닉스] 뉴스 목록 수집 중...
  -> 30건 수집
[현대차] 뉴스 목록 수집 중...
  -> 30건 수집
[LG에너지솔루션] 뉴스 목록 수집 중...
  -> 30건 수집
[삼성바이오로직스] 뉴스 목록 수집 중...
  -> 30건 수집
[KB금융] 뉴스 목록 수집 중...
  -> 30건 수집
[HD현대중공업] 뉴스 목록 수집 중...
  -> 30건 수집
[한화에어로스페이스] 뉴스 목록 수집 중...
  -> 30건 수집
[두산에너빌리티] 뉴스 목록 수집 중...
  -> 30건 수집
[NAVER] 뉴스 목록 수집 중...
  -> 30건 수집

총 300건 저장 완료 -> news_data.json
